# Approach 1 (Avoid-Step Cutoff, **weighted-power sweep**) — Fit on averaged raw CE

Variant of `approach_1_avoid_step` that (1) **measures per-run fit quality** on the tail of the
fit region and (2) **varies the residual weight power** to improve poor fits.

Fitting still applies only to data **before the first step artifact**
(`cutoff_BN` = first BN ≥ 100 where `abs(CE(BN) − CE(BN−1)) > 0.01`; full data if none).

**Residual weight:** `weight = x**power`. We fit with `power=1.0` first (the original behaviour).
If the last-`TAIL_N`-point **RMSE** is above `RMSE_TAIL_THRESH`, we re-fit with the remaining
candidate powers in `WEIGHT_POWERS` and keep whichever gives the best tail fit (lowest RMSE,
tie-break highest R²).

**Fit quality (per run):** `R2_last50` and `RMSE_last50` — unweighted R² and RMSE on the last
`TAIL_N` points of the fit region (the tail near the asymptote `A`, which sets `CE_L` and IPA).
Note: these CE tails are nearly flat, so R² is dominated by noise and can be negative even for a
good fit — **RMSE is the reliable indicator here** and drives the sweep; R² is kept for reference.

**Input:** `prune_layers_ALL/p-percentage_{p}/batch_size_{bs}/averaged_runs_p_{p}_bs_{bs}.csv`

**Output:** `intermediate/approach_1_fit_params_bs_{bs}.csv` (now with `weight_power, R2_last50,
RMSE_last50, n_tail`) plus `ipa_summary_approach_1_avoid_step_weighted.csv`.

**IPA:** `abs(CE_o - CE_L) / learn_BN` where `CE_L = CE_o - 0.85*(CE_o - A)`.

In [69]:
# === Cell 1 — Config, imports, helpers ===
import os, glob, re
import numpy as np
import pandas as pd
from lmfit import Parameters, minimize
import warnings
warnings.filterwarnings("ignore")

# ── CONFIG ────────────────────────────────────────────────────────────────────────────
BN_STEP_MIN = 100    # don't check for steps before this BN
STEP_THRESH = 0.01   # abs(CE(BN) - CE(BN-1)) > STEP_THRESH triggers cutoff

# Weight-power sweep + tail fit-quality
WEIGHT_POWERS    = [1.0,1.5,2]   # residual weight = x**power; 1.0 first (= original behaviour)
TAIL_N           = 100                # evaluate fit quality on the last N points of the fit region
RMSE_TAIL_THRESH = 0.001              # last-TAIL_N RMSE above this => "not perfectly fitted" => sweep
# (R^2 is also recorded but is unreliable on these near-flat CE tails, so RMSE drives the sweep.)
# ───────────────────────────────────────────────────────────────────────────────

# Paths
BASE_DIR = r"C:\Users\Student\Desktop\Projects\research\physlab\SLP\SLP-MNIST\prune_layers_ALL"
OUT_DIR  = r"C:\Users\Student\Desktop\Projects\research\physlab\SLP\SLP-MNIST\IPA_methods\Approach_1\test_1\approach_1_avoid_step_weighted"
INTERMEDIATE_DIR = os.path.join(OUT_DIR, "intermediate")
os.makedirs(INTERMEDIATE_DIR, exist_ok=True)

BATCH_SIZES = [64, 1024, 60000]
CE_o = np.log(10)   # max CE for 10-class problem, ~2.302585

# Auto-detect pruning percentages from prune_layers_ALL/p-percentage_*/
p_dirs = glob.glob(os.path.join(BASE_DIR, "p-percentage_*"))
PRUNING_LEVELS = sorted([
    float(re.search(r"p-percentage_([\d.]+)", d).group(1))
    for d in p_dirs
])
print(f"Found {len(PRUNING_LEVELS)} pruning percentages: {PRUNING_LEVELS}")
print(f"CE_o = ln(10) = {CE_o:.6f}")
print(f"BN_STEP_MIN = {BN_STEP_MIN}   STEP_THRESH = {STEP_THRESH}")
print(f"WEIGHT_POWERS = {WEIGHT_POWERS}   TAIL_N = {TAIL_N}   RMSE_TAIL_THRESH = {RMSE_TAIL_THRESH}")

# Fit-function helpers (verbatim from fitting_function_IPA.ipynb)
A_MIN, A_MAX = 0.1, 2.3
B_MIN, B_MAX = 0, 1000
N_MIN, N_MAX = 0.5, 2


def initialize_guesses(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    y = y[np.isfinite(y)]
    A0 = np.percentile(y, 5)
    B0 = np.percentile(y, 95) - A0
    n0 = 0.5
    if len(x) > 10:
        denom = y[0] - A0
        if abs(denom) > 1e-10:
            frac = max(1e-6, (y[0] - y[-1]) / denom)
            if frac > 0:
                n0 = max(0.3, min(1.5, -np.log(frac)))
    return A0, n0, B0


def model(params, x):
    vals = params.valuesdict()
    A, B, n = vals["A"], vals["B"], vals["n"]
    return A + B / ((x + 1) ** n)


def residual(params, x, data, power=1.0):
    weight = x ** power
    return weight * (model(params, x) - data)


def fit_curve(x, y, power=1.0):
    """Single fit at a given residual weight power."""
    mask  = ~np.isnan(y)
    x_fit = np.asarray(x)[mask]
    y_fit = np.asarray(y)[mask]
    if len(x_fit) < 10:
        return None
    A0, n0, B0 = initialize_guesses(x_fit, y_fit)
    params = Parameters()
    params.add("A", value=A0, min=A_MIN, max=A_MAX)
    params.add("B", value=B0, min=B_MIN, max=B_MAX)
    params.add("n", value=n0, min=N_MIN, max=N_MAX)
    try:
        return minimize(residual, params, args=(x_fit, y_fit), kws={"power": power})
    except Exception:
        return None


def fit_quality_tail(x, y, A, B, n, tail_n=TAIL_N):
    """Unweighted R^2 and RMSE on the last `tail_n` points of the fit region."""
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    m = np.isfinite(x) & np.isfinite(y)
    x, y = x[m], y[m]
    k = int(min(tail_n, len(x)))
    if k < 2:
        return np.nan, np.nan, k
    xt, yt = x[-k:], y[-k:]
    yhat   = A + B / ((xt + 1) ** n)
    resid  = yhat - yt
    rmse   = float(np.sqrt(np.mean(resid ** 2)))
    ss_res = float(np.sum(resid ** 2))
    ss_tot = float(np.sum((yt - yt.mean()) ** 2))
    r2     = float(1 - ss_res / ss_tot) if ss_tot > 0 else np.nan
    return r2, rmse, k


def _tail_better(cand, best):
    """Better tail fit = lower last-50 RMSE; tie-break higher R^2."""
    crm = cand["RMSE_last50"] if np.isfinite(cand["RMSE_last50"]) else np.inf
    brm = best["RMSE_last50"] if np.isfinite(best["RMSE_last50"]) else np.inf
    if crm != brm:
        return crm < brm
    cr = cand["R2_last50"] if np.isfinite(cand["R2_last50"]) else -np.inf
    br = best["R2_last50"] if np.isfinite(best["R2_last50"]) else -np.inf
    return cr > br


def fit_curve_sweep(x, y, powers=WEIGHT_POWERS, tail_n=TAIL_N, rmse_thresh=RMSE_TAIL_THRESH):
    """Fit at the first power; if its last-`tail_n` RMSE <= rmse_thresh, keep it (= original
    behaviour). Otherwise re-fit with the remaining candidate powers and keep the best tail
    fit (lowest RMSE, tie-break highest R^2). Returns dict(result, A, B, n, weight_power,
    R2_last50, RMSE_last50, n_tail) or None if no power converged."""
    best = None
    for i, power in enumerate(powers):
        result = fit_curve(x, y, power=power)
        if result is None:
            continue
        A = result.params["A"].value
        B = result.params["B"].value
        n = result.params["n"].value
        r2, rmse, k = fit_quality_tail(x, y, A, B, n, tail_n=tail_n)
        cand = {"result": result, "A": A, "B": B, "n": n, "weight_power": power,
                "R2_last50": r2, "RMSE_last50": rmse, "n_tail": k}
        if best is None or _tail_better(cand, best):
            best = cand
        # default power already good enough -> don't bother sweeping
        if i == 0 and np.isfinite(rmse) and rmse <= rmse_thresh:
            break
    return best


# --- IPA: learn_BN from avg data; analytical fallback when data ends early ---
# CE_L = CE_o - 0.85*(CE_o - A)   -- derived from fitted A
# IPA  = abs(CE_o - CE_L) / learn_BN
#
# learn_BN priority:
#   1. Truncated averaged data: first BN where Avg_CE_Test <= CE_L
#   2. Analytical fallback: learn_BN = ceil((B / (CE_L - A))^(1/n) - 1)
def compute_ipa_from_fit(x_grid, ce_data, A, B, n):
    x_grid  = np.asarray(x_grid,  dtype=float)
    ce_data = np.asarray(ce_data, dtype=float)
    CE_L = CE_o - 0.90 * (CE_o - A)
    mask = ce_data <= CE_L

    if mask.any():
        learn_BN        = float(x_grid[mask][0])
        avg_CE_at_learn = float(ce_data[mask][0])
    else:
        denom = CE_L - A
        if denom <= 0 or n <= 0 or B <= 0:
            return {"CE_L": CE_L, "learn_BN": np.nan, "IPA": np.nan, "avg_CE_learn_at_BN": np.nan}
        BN_analytic = (B / denom) ** (1.0 / n) - 1.0
        if not np.isfinite(BN_analytic) or BN_analytic <= 0:
            return {"CE_L": CE_L, "learn_BN": np.nan, "IPA": np.nan, "avg_CE_learn_at_BN": np.nan}
        learn_BN        = float(np.ceil(BN_analytic))
        avg_CE_at_learn = np.nan

    if learn_BN == 0:
        return {"CE_L": CE_L, "learn_BN": 0.0, "IPA": np.nan, "avg_CE_learn_at_BN": avg_CE_at_learn}
    IPA = abs(CE_o - CE_L) / learn_BN
    return {"CE_L": CE_L, "learn_BN": learn_BN, "IPA": IPA, "avg_CE_learn_at_BN": avg_CE_at_learn}


print("Cell 1 ready.")

Found 19 pruning percentages: [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.82, 0.84, 0.86, 0.88, 0.9, 0.92, 0.94, 0.96, 0.98, 1.0]
CE_o = ln(10) = 2.302585
BN_STEP_MIN = 100   STEP_THRESH = 0.01
WEIGHT_POWERS = [1.0, 1.5, 2]   TAIL_N = 100   RMSE_TAIL_THRESH = 0.001
Cell 1 ready.


In [70]:
# === Cell 2 — Approach 1 (avoid-step, weighted sweep): detect step cutoff; fit on clean data ===
# For each (P%, BS):
#   1. Load full averaged CE data.
#   2. Find cutoff_BN = first BN >= BN_STEP_MIN where abs(CE(BN)-CE(BN-1)) > STEP_THRESH.
#      If no step found, use all data (cutoff_BN = max BN in data).
#   3. Fit A,B,n on BN < cutoff_BN via fit_curve_sweep (weight = x**power, best tail fit).
#   4. Record per-run fit quality (R2_last50, RMSE_last50) and the chosen weight_power.
#   5. Find learn_BN from truncated avg data; analytical fallback if needed.
inter_by_bs = {}

for bs in BATCH_SIZES:
    print("\n" + "=" * 70)
    print(f"  Approach 1 (avoid-step, weighted) — Batch size {bs}")
    print("=" * 70)
    rows = []
    for p in PRUNING_LEVELS:
        avg_csv = os.path.join(BASE_DIR, f"p-percentage_{p}", f"batch_size_{bs}",
                               f"averaged_runs_p_{p}_bs_{bs}.csv")
        if not os.path.exists(avg_csv):
            print(f"  [SKIP] P%={p*100:5.1f}%  — missing {avg_csv}")
            continue
        df = pd.read_csv(avg_csv)
        df.columns = df.columns.str.strip()
        ce_col = next((c for c in df.columns if c in ("Avg_CE_Test", "Avg_CE_test")), None)
        bn_col = next((c for c in df.columns if "Batch" in c), None)
        if ce_col is None or bn_col is None:
            print(f"  [SKIP] P%={p*100:5.1f}%  — unexpected columns {list(df.columns)}")
            continue
        df = df.dropna(subset=[ce_col, bn_col]).reset_index(drop=True)

        # ── Step detection ──────────────────────────────────────────────────────────────────
        bns = df[bn_col].values.astype(float)
        ces = df[ce_col].values.astype(float)
        cutoff_BN     = float(bns[-1])   # default: use all data
        step_detected = False
        for i in range(1, len(bns)):
            if bns[i] >= BN_STEP_MIN:
                if abs(ces[i] - ces[i - 1]) > STEP_THRESH:
                    cutoff_BN     = float(bns[i])
                    step_detected = True
                    break
        # ────────────────────────────────────────────────────────────────────────────────

        df_fit = df[df[bn_col] < cutoff_BN]   # exclude step point itself
        x = df_fit[bn_col].values.astype(float)
        y = df_fit[ce_col].values.astype(float)

        fit = fit_curve_sweep(x, y)
        if fit is None:
            print(f"  [FAIL] P%={p*100:5.1f}%  — fit did not converge")
            continue
        A       = fit["A"]
        B       = fit["B"]
        n       = fit["n"]
        wp      = fit["weight_power"]
        r2_50   = fit["R2_last50"]
        rmse_50 = fit["RMSE_last50"]
        n_tail  = fit["n_tail"]
        ipa = compute_ipa_from_fit(x, y, A, B, n)

        step_tag = "[step detected]" if step_detected else "[no step, full data]"
        src      = "data" if np.isfinite(ipa["avg_CE_learn_at_BN"]) else "analytic"
        print(f"  P%={p*100:5.1f}%  cutoff_BN={cutoff_BN:>6.0f} {step_tag:<22}  "
              f"pow={wp:>3}  R2_50={r2_50:7.4f}  RMSE_50={rmse_50:7.4f}  "
              f"A={A:.4f}  CE_L={ipa['CE_L']:.4f}  learn_BN={ipa['learn_BN']!r:>8}  "
              f"IPA={ipa['IPA']}  [{src}]")

        rows.append({
            "P%":                 p * 100,
            "A":                  A,
            "B":                  B,
            "n":                  n,
            "CE_o":               CE_o,
            "CE_L":               ipa["CE_L"],
            "learn_BN":           ipa["learn_BN"],
            "avg_CE_learn_at_BN": ipa["avg_CE_learn_at_BN"],
            "IPA":                ipa["IPA"],
            "cutoff_BN":          cutoff_BN,
            "weight_power":       wp,
            "R2_last50":          r2_50,
            "RMSE_last50":        rmse_50,
            "n_tail":             n_tail,
        })

    if rows:
        bs_df = pd.DataFrame(rows)
        inter_path = os.path.join(INTERMEDIATE_DIR, f"approach_1_fit_params_bs_{bs}.csv")
        bs_df.to_csv(inter_path, index=False)
        print(f"  Saved: {inter_path}")
        inter_by_bs[bs] = bs_df

print("\n[Cell 2 done]")


  Approach 1 (avoid-step, weighted) — Batch size 64
  P%=  0.0%  cutoff_BN=   263 [step detected]         pow=1.5  R2_50= 0.7390  RMSE_50= 0.0033  A=0.3002  CE_L=0.5004  learn_BN=    41.0  IPA=0.0439549965740334  [data]
  P%= 10.0%  cutoff_BN=   273 [step detected]         pow=  2  R2_50= 0.7940  RMSE_50= 0.0030  A=0.2994  CE_L=0.4997  learn_BN=    45.0  IPA=0.04006462697422831  [data]
  P%= 20.0%  cutoff_BN=   272 [step detected]         pow=  2  R2_50= 0.8541  RMSE_50= 0.0027  A=0.2900  CE_L=0.4912  learn_BN=    52.0  IPA=0.034833783278586265  [data]
  P%= 30.0%  cutoff_BN=   261 [step detected]         pow=  2  R2_50= 0.9448  RMSE_50= 0.0019  A=0.2898  CE_L=0.4911  learn_BN=    57.0  IPA=0.031780481060557954  [data]
  P%= 40.0%  cutoff_BN=   285 [step detected]         pow=1.0  R2_50= 0.8875  RMSE_50= 0.0026  A=0.3048  CE_L=0.5046  learn_BN=    61.0  IPA=0.029475180485454652  [data]
  P%= 50.0%  cutoff_BN=   322 [step detected]         pow=  2  R2_50= 0.8515  RMSE_50= 0.0025  A=0.2

In [71]:
# === Cell 3 — Build wide summary CSV for Approach 1 (avoid-step, weighted) ===
# Schema: P%, IPA_Avg_64, STD_64, IPA_Avg_1024, STD_1024, IPA_Avg_60000, STD_60000
# STD columns blank (NaN) for single-curve approaches.
summary_rows = []
for p in PRUNING_LEVELS:
    row = {"P%": p * 100}
    for bs in BATCH_SIZES:
        df = inter_by_bs.get(bs)
        if df is None:
            mean_val = np.nan
        else:
            sub = df[df["P%"] == p * 100]
            mean_val = float(sub["IPA"].iloc[0]) if not sub.empty else np.nan
        row[f"IPA_Avg_{bs}"] = mean_val
        row[f"STD_{bs}"]     = np.nan
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows, columns=[
    "P%", "IPA_Avg_64", "STD_64", "IPA_Avg_1024", "STD_1024", "IPA_Avg_60000", "STD_60000"
])
out_csv = os.path.join(OUT_DIR, "ipa_summary_approach_1_avoid_step_weighted.csv")
summary_df.to_csv(out_csv, index=False)
print(f"\nFinal summary written: {out_csv}")
print(summary_df.to_string(index=False))


Final summary written: C:\Users\Student\Desktop\Projects\research\physlab\SLP\SLP-MNIST\IPA_methods\Approach_1\test_1\approach_1_avoid_step_weighted\ipa_summary_approach_1_avoid_step_weighted.csv
   P%  IPA_Avg_64  STD_64  IPA_Avg_1024  STD_1024  IPA_Avg_60000  STD_60000
  0.0    0.043955     NaN      0.064985       NaN       0.066870        NaN
 10.0    0.040065     NaN      0.057109       NaN       0.066447        NaN
 20.0    0.034834     NaN      0.052227       NaN       0.060040        NaN
 30.0    0.031780     NaN      0.049205       NaN       0.053206        NaN
 40.0    0.029475     NaN      0.042393       NaN       0.046485        NaN
 50.0    0.023758     NaN      0.033956       NaN       0.040282        NaN
 60.0    0.018097     NaN      0.029746       NaN       0.032848        NaN
 70.0    0.015666     NaN      0.022745       NaN       0.028169        NaN
 80.0    0.009149     NaN      0.015837       NaN       0.018615        NaN
 82.0    0.010744     NaN      0.015273    

In [72]:
# === Cell 4 — Plot IPA vs P% ===
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

TAG   = "1_avoid_step_weighted"
TITLE = "Approach 1 (avoid-step, weighted-power sweep) — Fit on averaged raw CE"
BS_COLOR = {64: "#1f77b4", 1024: "#d62728", 60000: "#2ca02c"}

plt.rcParams.update({"font.size": 14})
fig, ax = plt.subplots(figsize=(10, 6))

for bs in BATCH_SIZES:
    mean_col = f"IPA_Avg_{bs}"
    sub = summary_df.dropna(subset=[mean_col])
    if sub.empty:
        continue
    ax.plot(sub["P%"].values, sub[mean_col].values,
            label=f"BS={bs}", color=BS_COLOR[bs], marker="o", markersize=6, linewidth=2)

ax.set_xlabel("Pruning Percentage (%)")
ax.set_ylabel("IPA")
ax.set_title(TITLE, fontsize=14)
ax.grid(True, which="both", alpha=0.3)
ax.legend(frameon=False)

out_png = os.path.join(OUT_DIR, f"ipa_plot_approach_{TAG}.png")
plt.tight_layout()
plt.savefig(out_png, dpi=300, bbox_inches="tight")
plt.close(fig)
print(f"Saved: {out_png}")

Saved: C:\Users\Student\Desktop\Projects\research\physlab\SLP\SLP-MNIST\IPA_methods\Approach_1\test_1\approach_1_avoid_step_weighted\ipa_plot_approach_1_avoid_step_weighted.png
